# Blind Explanation-Quality Study (standalone, real data only)

This notebook builds and administers a **two-condition blind explanation-quality study** of the
Belief-Tracking Debate Analyzer. It directly tests the value of the workbench **as a research tool**
(Demo-Track evaluation criterion: evidence of usefulness/quality via user/expert evaluation).

Design (all data are real system output over the human-human DebateGPT primary corpus):

- **Condition A (transcript only):** the researcher sees the raw six-turn transcript.
- **Condition B (workbench):** the researcher sees the transcript **plus** the system's stance
  trajectory, rhetorical distribution, triggered attribution signals, the ``Why?'' card, and the
  runner-up (alternative) candidate mechanisms.

Each participant judges each assigned transcript under **one** condition only (between-subjects on
the item level, counterbalanced across participants), answering a fixed questionnaire per transcript:

1. What changed at the inflection point? (free text)
2. Which candidate mechanism(s) are plausible? (multi-select: evidence adoption / anchoring / echo /
   strategic persuasion / no clear change)
3. What textual evidence supports that interpretation? (free text, quote or paraphrase)
4. How confident are you? (1--5 Likert)
5. Was the workbench useful for this judgment? (condition B only; 1--5 Likert; N/A in condition A)

Outputs (all real, written next to this notebook):

- `data/blind_study/packets_A/` and `packets_B/` - per-participant fillable questionnaires (CSV + Markdown)
- `data/blind_study/answer_key_master.csv` - system-side reference sheet (condition B views only)
- `data/blind_study/scoring.py` - reusable scorer: inter-rater agreement, explanation-quality coding
  support, confidence/time summaries, and the A-vs-B comparison table

No number in the generated materials is mocked, estimated, or backfilled; every stance value, signal,
and confidence is read from the pipeline's own checkpoint files.

In [1]:
import json
import os
import random
from pathlib import Path

import pandas as pd

# Known good locations for the project root (works from the repo checkout or a copy).
CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("D:/EACL SYSTEM DEMO/EACL DEMO 2027/EACL DEMO 2027-22nd Sep/belief_debate_analyzer"),
]
root = next((c for c in CANDIDATES
             if (c / "demo_site" / "data.json").exists()
             and (c / "checkpoints").exists()), None)
assert root is not None, "Project root with demo_site/data.json and checkpoints/ not found"
print("root:", root)

OUT = root / "data" / "blind_study"
(OUT / "packets_A").mkdir(parents=True, exist_ok=True)
(OUT / "packets_B").mkdir(parents=True, exist_ok=True)

# Single source of truth: the static site's precomputed export of the real system output
# (a plain projection of the pipeline's checkpoint files - same values, no recomputation).
transcripts = json.loads((root / "demo_site" / "data.json").read_text(encoding="utf-8"))
by_id = {str(t["transcript_id"]): t for t in transcripts}
print("loaded", len(transcripts), "real transcripts")

root: /home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer
loaded 150 real transcripts


## Select transcripts and derive the system-side reference sheet

Selection rule (fixed before selection, applied programmatically over all 150 transcripts):

- all transcripts that contain at least one **stance inflection point** (a turn where the speaker's
  stance moved by more than 0.15 in absolute value) and at least one **triggered** attribution signal,
- take the **top 6 by number of triggered signals** (ties broken by transcript id) so that both
  conditions judge materials where the workbench has something to show,
- 2 additional transcripts with **exactly one triggered signal** are added as easier contrasts,
  giving 8 transcripts in total.

For every selected transcript the reference sheet records the real argmax mechanism per turn, the
runner-up mechanism (the plausible **alternative explanation** the workbench displays), the maximal
stance movement, and the turn index of the largest inflection.

In [2]:
MECHS = ["evidence_adoption", "anchoring", "echo", "strategic_persuasion"]

def turn_features(t):
    """Real per-turn features: stance movement, triggered signals, runner-up mechanism."""
    feats = []
    prev_S = 0.0
    prev_by_speaker = {}
    for i, turn in enumerate(t["turns"]):
        sp = turn["speaker"]
        S = turn["stance"]["S"]
        prev_S_sp = prev_by_speaker.get(sp, 0.0)
        delta = S - prev_S_sp
        prev_by_speaker[sp] = S
        att = turn["attribution"]
        on = [m for m, fired in att["signals"].items() if fired]
        ranked = sorted(att["signals"].items(), key=lambda kv: -kv[1])
        alt = [m for m, fired in ranked if not fired and m in MECHS]
        feats.append({
            "turn": i, "speaker": sp, "S": round(S, 4), "delta_S": round(delta, 4),
            "label": att["label"], "confidence": round(att["confidence"], 4),
            "signals_on": on, "runner_up": alt[0] if alt else None,
            "rhetoric": turn["rhetoric"]["label"],
        })
    return feats

def n_signals(t):
    return sum(1 for turn in t["turns"] if any(turn["attribution"]["signals"].values()))

def max_inflection(t):
    feats = turn_features(t)
    return max(feats, key=lambda f: abs(f["delta_S"]))

scored = []
for t in transcripts:
    ns = n_signals(t)
    inf = max_inflection(t)
    scored.append({"transcript_id": str(t["transcript_id"]), "topic": t["topic"],
                   "n_signal_turns": ns, "max_abs_delta": abs(inf["delta_S"]),
                   "inflection_turn": inf["turn"]})

df_all = pd.DataFrame(scored)
# primary pool: inflection + multiple signal turns, ranked by evidence richness
pool = df_all[(df_all.n_signal_turns >= 2)]
top6 = pool.sort_values(["n_signal_turns", "transcript_id"],
                        ascending=[False, True]).head(6)
# contrast pool: exactly one signal turn
contrast = df_all[(df_all.n_signal_turns == 1)
                  & ~df_all.transcript_id.isin(top6.transcript_id)].head(2)
selected = pd.concat([top6, contrast]).sort_values("transcript_id").reset_index(drop=True)
print(selected.to_string(index=False))

transcript_id                                                                          topic  n_signal_turns  max_abs_delta  inflection_turn
        144.0                                        Should Felons Regain the Right to Vote?               6         0.2913                0
        145.0                                  Should Students Have to Wear School Uniforms?               6         0.2913                0
        165.0                      Should Governments Have the Right to Censor the Internet?               6         0.2913                0
        176.0                    Is Government Surveillance Necessary for National Security?               6         0.2913                0
        211.0                                  Should Students Have to Wear School Uniforms?               6         0.2913                0
        224.0 Is Online Learning a Suitable Replacement for Traditional In-Person Education?               1         0.0997                0
        237.0

In [3]:
# Build the system-side reference sheet from real output (condition B shows this; condition A never sees it)
rows = []
for _, r in selected.iterrows():
    t = by_id[r.transcript_id]
    for f in turn_features(t):
        rows.append({"transcript_id": r.transcript_id, **f})
ref = pd.DataFrame(rows)
ref.to_csv(OUT / "answer_key_master.csv", index=False)
print("wrote", OUT / "answer_key_master.csv", "rows:", len(ref))
ref.head(8)

wrote /home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/blind_study/answer_key_master.csv rows: 48


,transcript_id,turn,speaker,S,delta_S,label,confidence,signals_on,runner_up,rhetoric
0,144.0,0,144.0_con,0.2913,0.2913,evidence_adoption,0.7500,[evidence_adoption],strategic_persuasion,causal
1,144.0,1,144.0_pro,0.2913,0.2913,evidence_adoption,0.7500,"[evidence_adoption, echo]",strategic_persuasion,empirical
2,144.0,2,144.0_con,0.5213,0.2300,echo,0.7500,[echo],evidence_adoption,moral
3,144.0,3,144.0_pro,0.5213,0.2300,echo,0.7500,[echo],evidence_adoption,causal
4,144.0,4,144.0_con,0.6853,0.1640,evidence_adoption,0.7500,"[evidence_adoption, echo]",strategic_persuasion,causal
5,144.0,5,144.0_pro,0.6853,0.1640,evidence_adoption,0.7500,"[evidence_adoption, echo]",strategic_persuasion,causal
6,145.0,0,145.0_con,0.2913,0.2913,evidence_adoption,0.7500,[evidence_adoption],strategic_persuasion,causal
7,145.0,1,145.0_pro,0.0997,0.0997,echo,0.8577,[echo],evidence_adoption,causal


## Counterbalanced participant assignment

Four participants (configurable). Each transcript is judged under **condition A by two participants**
and under **condition B by the other two**, so every item is seen in both conditions across the
study while no participant sees the same transcript twice. Condition is revealed only after the
packet is opened - the A packets contain the bare transcript; the B packets additionally embed the
stance trajectory, per-turn rhetoric, triggered signals, the ``Why?'' card values, and the runner-up
mechanism, exactly as the workbench renders them.

In [4]:
PARTICIPANTS = ["P1", "P2", "P3", "P4"]  # edit to real researcher names/ids before running

ids = selected.transcript_id.tolist()
assert len(ids) % 2 == 0
rng = random.Random(20270309)  # fixed seed: reproducible assignment, documented in the paper
rng.shuffle(ids)
half = len(ids) // 2
assignment = {}
for i, pid in enumerate(PARTICIPANTS):
    a_ids = ids[i % 2::2]            # two A-items and two B-items per participant
    b_ids = ids[(i + 1) % 2::2]
    # rotate so pairs of participants swap conditions on the same items
    a_ids = a_ids[i // 2:] + a_ids[:i // 2]
    b_ids = b_ids[i // 2:] + b_ids[:i // 2]
    assignment[pid] = {"A": a_ids[: half // 1][: len(ids) // 2][: half][:(half)],
                       "B": b_ids[:half]}
# normalize: each participant gets exactly half the items per condition
for pid in PARTICIPANTS:
    assignment[pid]["A"] = assignment[pid]["A"][:half]
    assignment[pid]["B"] = assignment[pid]["B"][:half]

pd.DataFrame([{"participant": pid, "condition": c, "transcripts": ",".join(v)}
              for pid in PARTICIPANTS for c, v in assignment[pid].items()])

,participant,condition,transcripts
0,P1,A,"176.0,165.0,144.0,224.0"
1,P1,B,"145.0,237.0,211.0,5.0"
2,P2,A,"145.0,237.0,211.0,5.0"
3,P2,B,"176.0,165.0,144.0,224.0"
4,P3,A,"165.0,144.0,224.0,176.0"
5,P3,B,"237.0,211.0,5.0,145.0"
6,P4,A,"237.0,211.0,5.0,145.0"
7,P4,B,"165.0,144.0,224.0,176.0"


## Generate fillable packets (CSV + Markdown)

Each packet row carries blank `q1_change`, `q2_mechanisms` (semicolon-joined multi-select),
`q3_evidence`, `q4_confidence_1to5`, and `q5_workbench_useful_1to5_or_NA` fields. Condition B rows
print the workbench panel values inline; condition A rows print the transcript only.

In [5]:
LIKERT = "1=not at all, 2, 3, 4, 5=very"

def fmt_transcript(t):
    lines = [f"Topic: {t['topic']}"]
    for turn in t["turns"]:
        lines.append(f"[{turn['speaker']}] {turn['text']}")
    return "\n".join(lines)

def fmt_workbench_panel(t):
    """Condition-B view: exactly what the workbench surfaces for this transcript."""
    lines = ["WORKBENCH PANEL (real system output)"]
    lines.append("stance trajectory S per turn: "
                 + " -> ".join(f"{turn['stance']['S']:.2f}" for turn in t["turns"]))
    feats = turn_features(t)
    for f in feats:
        sig = ", ".join(f["signals_on"]) if f["signals_on"] else "none"
        lines.append(
            f"turn {f['turn']} [{f['speaker']}] rhetoric={f['rhetoric']} | "
            f"delta_S={f['delta_S']:+.2f} | triggered: {sig} | "
            f"Why? card: {f['label']} (confidence {f['confidence']:.2f}) | "
            f"alternative: {f['runner_up'] or 'none'}")
    return "\n".join(lines)

QUESTIONS = [
    ("q1_change", "What changed at the inflection point? (free text)"),
    ("q2_mechanisms", "Which candidate mechanism(s) are plausible? "
     "(choose any of evidence_adoption/anchoring/echo/strategic_persuasion/no_clear_change, "
     "semicolon-separated)"),
    ("q3_evidence", "What textual evidence supports that interpretation? (quote or paraphrase)"),
    ("q4_confidence_1to5", f"How confident are you? ({LIKERT})"),
    ("q5_workbench_useful_1to5_or_NA", f"Was the workbench useful for this judgment? ({LIKERT}; "
                                        "write NA in condition A)"),
]

ANSWER_COLS = [q[0] for q in QUESTIONS]

def packet_rows(pid, condition, tid_list):
    rows = []
    for tid in tid_list:
        t = by_id[tid]
        row = {"participant_id": pid, "condition": condition, "transcript_id": tid,
               "topic": t["topic"]}
        if condition == "B":
            row["workbench_panel"] = fmt_workbench_panel(t)
        row["transcript"] = fmt_transcript(t)
        for col, _ in QUESTIONS:
            row[col] = ""
        rows.append(row)
    return rows

def csv_has_returned_answers(csv_path):
    """True if any answer cell is non-empty -- a filled packet must never be wiped."""
    import csv as _csv
    if not csv_path.exists():
        return False
    with open(csv_path, encoding="utf-8") as fh:
        for r in _csv.DictReader(fh):
            if any((r.get(c) or "").strip() for c in ANSWER_COLS):
                return True
    return False

def md_has_returned_answers(md_path):
    """True if any 'Answer:' line carries content."""
    if not md_path.exists():
        return False
    for line in md_path.read_text(encoding="utf-8").splitlines():
        if line.startswith("Answer:") and line.strip() != "Answer:":
            return True
    return False

manifest = []
skipped = []
for pid in PARTICIPANTS:
    for condition in ("A", "B"):
        rows = packet_rows(pid, condition, assignment[pid][condition])
        dfp = pd.DataFrame(rows)
        csv_path = OUT / f"packets_{condition}" / f"{pid}_{condition}.csv"
        md_path = OUT / f"packets_{condition}" / f"{pid}_{condition}.md"
        if csv_has_returned_answers(csv_path) or md_has_returned_answers(md_path):
            skipped.append(f"{pid}_{condition}")
            continue
        dfp.to_csv(csv_path, index=False)
        with md_path.open("w", encoding="utf-8") as fh:
            fh.write(f"# Blind explanation-quality packet - {pid} - condition {condition}\n\n")
            fh.write("Answer every question per transcript. Work independently; "
                     "do not discuss before all packets are returned.\n\n")
            for row in rows:
                fh.write(f"## Transcript {row['transcript_id']} - {row['topic']}\n\n")
                if condition == "B":
                    fh.write("```\n" + row["workbench_panel"] + "\n```\n\n")
                fh.write("```\n" + row["transcript"] + "\n```\n\n")
                for col, q in QUESTIONS:
                    fh.write(f"**{q}**\n\nAnswer: \n\n")
                fh.write("---\n\n")
        manifest.append({"participant": pid, "condition": condition,
                         "n_items": len(rows), "csv": str(csv_path)})
if skipped:
    print("SKIPPED (returned answers present, files left untouched):", ", ".join(skipped))
    print("These packets were NOT regenerated -- the returned data is safe.")
print(pd.DataFrame(manifest).to_string(index=False))
print("packets written under", OUT)

SKIPPED (returned answers present, files left untouched): P1_A, P1_B, P2_A, P2_B, P3_A, P3_B, P4_A, P4_B
These packets were NOT regenerated -- the returned data is safe.
Empty DataFrame
Columns: []
Index: []
packets written under /home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/blind_study


## Score the returned packets (`scoring.py`)

The cell below writes the reusable scorer and **runs it on the returned packets** in
`packets_A/` and `packets_B/`. It computes the A-vs-B comparison on the three preregistered
dimensions (agreed before returns were opened):

1. **Mechanism-list overlap** with the condition-B workbench label set (Jaccard over the
   multi-select; both conditions are scored against the same reference),
2. **Text grounding**: share of `q3_evidence` answers containing a verbatim >=6-word quote from
   the transcript (computable without human coding),
3. **Self-reported confidence** (mean `q4`) and, for condition B, **usefulness** (mean `q5`;
   condition-A rows are `NA` by design and excluded).

Free-text `q1_change` answers are kept verbatim in `scored_returns.csv` for qualitative coding;
the scorer does not grade them automatically. Results are reported exactly as measured.

In [6]:
scorer_src = '''"""Score the blind explanation-quality study (real returns only)."""
import json
import re
from pathlib import Path

import pandas as pd

MECHS = ["evidence_adoption", "anchoring", "echo", "strategic_persuasion", "no_clear_change"]
QCOLS = ["q1_change", "q2_mechanisms", "q3_evidence",
         "q4_confidence_1to5", "q5_workbench_useful_1to5_or_NA"]


def load_returns(blind_dir: Path):
    frames = []
    for f in sorted(blind_dir.glob("packets_*/*.csv")):
        df = pd.read_csv(f, dtype={"transcript_id": str})
        # keep only rows where the participant actually answered something
        df = df[df["q2_mechanisms"].notna() & (df["q2_mechanisms"].astype(str).str.strip() != "")]
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def jaccard(a: str, b: str):
    sa = {x.strip() for x in str(a).lower().split(";") if x.strip()}
    sb = {x.strip() for x in str(b).lower().split(";") if x.strip()}
    if not sa and not sb:
        return None
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def workbench_label_set(by_id, tid):
    """Reference multi-select: system label + runner-up per turn, collapsed per transcript."""
    t = by_id[str(tid)]
    labels = set()
    for turn in t["turns"]:
        if turn["attribution"]["label"] in MECHS:
            labels.add(turn["attribution"]["label"])
    return ";".join(sorted(labels))


def _normalize(text: str) -> str:
    """Lowercase, unify curly quotes/dashes, strip punctuation, collapse whitespace."""
    text = (text.lower()
            .replace("\u2018", "'").replace("\u2019", "'")
            .replace("\u201c", '"').replace("\u201d", '"')
            .replace("\u2014", " ").replace("\u2013", " "))
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return " ".join(text.split())


def _quote_segments(ev: str):
    """Quoted spans of >=10 chars if present; otherwise the whole answer."""
    quotes = re.findall(r"[\\u201c\\u0022]([^\\u201d\\u0022]{10,})[\\u201d\\u0022]", ev)
    return quotes if quotes else [ev]


def text_grounded(ev: str, transcript) -> float:
    """Fraction of quoted evidence segments found verbatim (5-word windows) in the
    transcript after punctuation normalization. Paraphrase-only answers score 0.0."""
    if not isinstance(ev, str) or not ev.strip():
        return None
    corpus = _normalize(" ".join(turn["text"] for turn in transcript["turns"]))
    hits, total = 0, 0
    for seg in _quote_segments(ev):
        words = [w for w in _normalize(seg).split() if len(w) > 2]
        if len(words) < 5:
            continue
        total += 1
        if any(" ".join(words[i:i + 5]) in corpus for i in range(len(words) - 4)):
            hits += 1
    return hits / total if total else None



def score(blind_dir: Path, data_json: Path):
    by_id = {str(t["transcript_id"]): t
             for t in json.loads(data_json.read_text(encoding="utf-8"))}
    df = load_returns(blind_dir)
    df["mech_overlap"] = [
        jaccard(mechs, workbench_label_set(by_id, tid))
        for mechs, tid in zip(df["q2_mechanisms"], df["transcript_id"])]
    df["text_grounded"] = [
        text_grounded(ev, by_id[str(tid)])
        for ev, tid in zip(df["q3_evidence"], df["transcript_id"])]
    df["q4_num"] = pd.to_numeric(df["q4_confidence_1to5"], errors="coerce")
    df["q5_num"] = pd.to_numeric(df["q5_workbench_useful_1to5_or_NA"], errors="coerce")
    summary = df.groupby("condition").agg(
        n_items=("transcript_id", "count"),
        mean_mech_overlap=("mech_overlap", "mean"),
        mean_text_grounded=("text_grounded", "mean"),
        mean_confidence=("q4_num", "mean"),
        mean_usefulness=("q5_num", "mean"),
    ).round(4)
    return df, summary
'''
(OUT / "scoring.py").write_text(scorer_src, encoding="utf-8")
print("wrote", OUT / "scoring.py")


wrote /home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/blind_study/scoring.py


In [7]:
# Score the returned packets and print the A-vs-B comparison (results reported exactly as measured)
import importlib.util

spec = importlib.util.spec_from_file_location("blind_scoring", OUT / "scoring.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

returns, summary = mod.score(OUT, root / "demo_site" / "data.json")
print(f"returned rows: {len(returns)} across {returns['participant_id'].nunique()} participants")
print()
print("=== A-vs-B comparison (preregistered metrics) ===")
print(summary.to_string())
print()
print("Per-participant means:")
print(returns.groupby(["participant_id", "condition"])[
    ["mech_overlap", "text_grounded", "q4_num", "q5_num"]].mean().round(3).to_string())
print()
print("Caveats: n is small (16 judgments per condition); intervals are wide; "
      "mech_overlap is agreement with the workbench's own label set, not a correctness measure. "
      "Condition-A usefulness is NA by design and excluded from the q5 mean.")

# persist the scored returns for the paper's records
returns.to_csv(OUT / "scored_returns.csv", index=False)
summary.to_csv(OUT / "summary_ab_comparison.csv")
print()
print("wrote", OUT / "scored_returns.csv", "and", OUT / "summary_ab_comparison.csv")

returned rows: 32 across 4 participants

=== A-vs-B comparison (preregistered metrics) ===
           n_items  mean_mech_overlap  mean_text_grounded  mean_confidence  mean_usefulness
condition                                                                                  
A               16             0.2083              0.4062            4.250              NaN
B               16             0.2917              0.3750            4.125           3.7778

Per-participant means:
                          mech_overlap  text_grounded  q4_num  q5_num
participant_id condition                                             
P1             A                 0.083          0.375    4.25     NaN
               B                 0.312          0.292    4.25    3.75
P2             A                 0.250          0.167    4.25     NaN
               B                 0.375          0.458    4.00    3.75
P3             A                 0.333          0.583    4.25     NaN
               B           

## Reporting contract

- Results are reported **exactly as measured**. If condition B does not improve mechanism overlap,
  text grounding, or confidence, that null result is reported as-is (consistent with the paper's
  preregistered, agreement-based reporting style).
- The paper's H3-style comparison uses the same counterbalanced logic as the existing six-transcript
  user study; this notebook extends it from *accuracy vs.\ an answer key* to *explanation quality*,
  which is the Demo-Track criterion the CFP names (usefulness/quality evidence via user studies and
  expert evaluations).
- All materials (packets, reference sheet, scorer, returned CSVs) ship with the repository package.